<a href="https://colab.research.google.com/github/VictoriaO26/AI-assisted-Cloud-Infrastructure-Cybersecurity-Bootcamp/blob/main/Module%206%20/Network_Attack_Detection_with_Fairness_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IoT Network Attack Detection with Fairness Analysis

This workshop module examines how **data bias** and **algorithmic bias** in network intrusion detection can create opportunities for adversarial exploitation.

Students work through four sequential exercises:
1. **Detect bias and adversarial exposure** by examining class imbalance, group disparities, and fairness patterns in a biased network dataset.
2. **Mitigate data bias** with **IBM AI Fairness 360 (AIF360) Reweighing**, producing a less biased training set.
3. **Compare model behavior on the mitigated data** to study how different algorithms exhibit different fairness profiles and how biased decision boundaries can increase adversarial risk.
4. **Apply an additional bias-mitigation step** to further reduce algorithmic bias and narrow the scope for attacks that exploit biased predictions.

**Notebook focus:** This notebook operationalizes that sequence with fairness metrics, reweighing, multi-model comparison, and a postprocessing mitigation example on network metadata features. If Aequitas is used elsewhere in the workshop for auditing and visualization, the analysis here complements it by carrying the mitigation workflow forward in AIF360.

**Dataset:** CIC-BCCC-NRC TabularIoTAttack-2024 (`combined_sample.csv` — 50,000 samples)
**Protected Attribute:** Destination Port (`High_Port` — traffic on ports above vs below the median)
**Feature Selection:** Timing and metadata features (`Protocol`, `Flow Duration`, and IAT statistics) rather than the full 80-feature representation.

**Expected Research Skills Gain:** Learn how adversarial attacks can leverage data bias and algorithmic bias in network security pipelines, and how fairness-aware preprocessing, model comparison, and mitigation can reduce that risk.

**Students' Tasks:** Complete the exercise cells in order and keep the requested variable names.


## Section 1: Imports and Setup

This cell imports the libraries used for the hands-on fairness and mitigation workflow in Module 2: pandas and numpy for data preparation, matplotlib and seaborn for visualization, scikit-learn classifiers and evaluation metrics, and IBM's AIF360 toolkit for fairness measurement and mitigation. The notebook uses these tools to analyze how bias in network intrusion detection can affect model behavior and adversarial exposure.


In [ ]:
!pip -q install aif360 seaborn

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('aif360').setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import aif360
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric
from aif360.algorithms.preprocessing.reweighing import Reweighing

from tqdm import tqdm
from collections import OrderedDict

np.random.seed(42)
sns.set_style('whitegrid')
print(f'AIF360 version: {aif360.__version__}')

## Section 2: Helper Functions

This cell defines the `compute_metrics` helper function that calculates six key fairness metrics using AIF360's ClassificationMetric class: balanced accuracy, statistical parity difference (SPD), disparate impact (DI), average odds difference, equal opportunity difference, and Theil index. These metrics capture different dimensions of algorithmic fairness. This function is called repeatedly to evaluate each model's fairness on the test set.

In [ ]:
def compute_metrics(dataset_true, dataset_pred,
                    unprivileged_groups, privileged_groups, disp=True):
    """Compute key fairness metrics using AIF360 ClassificationMetric."""
    cm = ClassificationMetric(dataset_true, dataset_pred,
                              unprivileged_groups=unprivileged_groups,
                              privileged_groups=privileged_groups)
    metrics = OrderedDict()
    metrics['Balanced accuracy'] = 0.5 * (cm.true_positive_rate() + cm.true_negative_rate())
    metrics['Statistical parity difference'] = cm.statistical_parity_difference()
    metrics['Disparate impact'] = cm.disparate_impact()
    metrics['Average odds difference'] = cm.average_odds_difference()
    metrics['Equal opportunity difference'] = cm.equal_opportunity_difference()
    metrics['Theil index'] = cm.theil_index()

    if disp:
        for k, v in metrics.items():
            print(f'  {k} = {v:.4f}')

    return metrics

## Section 3: Load and Explore the Network Traffic Dataset

**Exercise 1 Goal:** Detect data bias, class imbalance, and group disparities in the biased network dataset before any mitigation is applied.


This cell loads the 50,000-sample network traffic dataset either from the notebook folder or by upload from the local computer. The dataset contains 90% benign traffic (45,000 samples) and 10% malicious traffic (5,000 attack samples) with 80 columns of network flow features including source/destination ports, protocol, flow duration, packet lengths, inter-arrival times, and TCP flags, plus the binary Label column (0=Benign, 1=Malicious).


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/acr-research-lab/AI4SecureCI/main/module_6/combined_sample.csv"

df = pd.read_csv(url)
print(f"Dataset shape: {df.shape}")
print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"\nLabel distribution:")
print(df["Label"].value_counts())
print(f"\nBenign rate: {(df['Label'] == 0).mean() * 100:.1f}%")
print(f"Malicious rate: {(df['Label'] == 1).mean() * 100:.1f}%")
df.head()

This cell examines the dataset's basic properties. It confirms the data types (mostly numeric features), checks for missing values, and provides descriptive statistics for all columns. This helps identify any data quality issues before proceeding with the fairness analysis.

In [ ]:
# Dataset info and statistics
print('Data types:')
print(df.dtypes.value_counts())
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\nBasic statistics:')
df.describe()

## Section 4: Exploratory Data Analysis

This cell creates a 2x3 grid of exploratory visualizations. The label distribution shows 45,000 benign (90%) vs 5,000 malicious (10%) samples. The protocol distribution confirms all traffic uses TCP (protocol 6). The destination port distribution shows two groups split at the median of 1883: the low-port group (≤1883) contains 36,735 samples across just 5 well-known service ports (80/HTTP, 443/HTTPS, 853/DNS-over-TLS, 1443, 1883/MQTT), while the high-port group (>1883) contains 13,265 samples across 2,156 unique ports including ephemeral ports (32000–65000+), 8883 (MQTT over TLS), and 6668 (IRC). Most critically, the bottom-right plot shows the **malicious rate by port group**: low-port traffic has only approximately 2.9% malicious rate while high-port traffic has approximately 29.8% malicious rate — a **27 percentage point gap** that represents the fairness challenge addressed in this module. This natural bias means models may learn to unfairly flag high-port traffic as malicious.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Label distribution
ax = axes[0, 0]
labels = df['Label'].value_counts()
ax.bar(['Benign (0)', 'Malicious (1)'], labels.values, color=['#2ecc71', '#e74c3c'])
ax.set_title('Label Distribution', fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(labels.values):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# 2. Protocol distribution
ax = axes[0, 1]
proto = df['Protocol'].value_counts()
ax.bar(proto.index.astype(str), proto.values, color='#3498db')
ax.set_title('Protocol Distribution', fontweight='bold')
ax.set_ylabel('Count')

# 3. Dst Port distribution (protected-attribute basis)
ax = axes[0, 2]
median_port = df['Dst Port'].median()
low_port = (df['Dst Port'] <= median_port).sum()
high_port = (df['Dst Port'] > median_port).sum()
ax.bar(['Low Port\n(<=median)', 'High Port\n(>median)'], [low_port, high_port], color=['#9b59b6', '#e67e22'])
ax.set_title(f'Dst Port Groups (median={median_port:.0f})', fontweight='bold')
ax.set_ylabel('Count')

# 4. Flow Duration distribution
ax = axes[1, 0]
ax.hist(df['Flow Duration'].clip(upper=df['Flow Duration'].quantile(0.95)), bins=50, color='#1abc9c', edgecolor='black')
ax.set_title('Flow Duration Distribution', fontweight='bold')
ax.set_xlabel('Duration')

# 5. Fwd IAT Mean distribution
ax = axes[1, 1]
ax.hist(df['Fwd IAT Mean'].clip(upper=df['Fwd IAT Mean'].quantile(0.95)), bins=50, color='#f39c12', edgecolor='black')
ax.set_title('Fwd IAT Mean Distribution', fontweight='bold')
ax.set_xlabel('Forward Inter-Arrival Time (Mean)')

# 6. Attack rate by port group (KEY FAIRNESS PLOT)
ax = axes[1, 2]
low_rate = df[df['Dst Port'] <= median_port]['Label'].mean() * 100
high_rate = df[df['Dst Port'] > median_port]['Label'].mean() * 100
bars = ax.bar(['Low Port\n(Privileged)', 'High Port\n(Unprivileged)'], [low_rate, high_rate], color=['#2ecc71', '#e74c3c'])
ax.set_title('Malicious Rate by Port Group', fontweight='bold')
ax.set_ylabel('Malicious Rate (%)')
for bar, rate in zip(bars, [low_rate, high_rate]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{rate:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Network Traffic Dataset — Exploratory Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Section 5: AIF360 Dataset Preparation

This section creates a protected attribute from the Destination Port and selects timing/metadata features
for classification. Using metadata features (rather than all 80 features) produces a more
realistic and challenging classification task where bias mitigation can have meaningful impact.

This cell creates the protected attribute and selects features for the fairness analysis.\n\n**Protected Attribute — High_Port:** The Destination Port is binarized at the dataset median of 1883. **Low port (≤1883, privileged group):** encompasses only 5 unique destination ports — port 80 (HTTP), 443 (HTTPS), 853 (DNS over TLS), 1443, and 1883 (MQTT, the standard IoT messaging protocol). These are well-known service ports associated with standard IoT device communication. This group contains 36,735 samples with only approximately 2.85% malicious traffic. **High port (>1883, unprivileged group):** encompasses 2,156 unique destination ports, including 8883 (MQTT over TLS), 6668 (IRC), 3478 (STUN/TURN), 5228, 8000, and thousands of ephemeral/dynamic ports in the 32000–65000+ range. These higher ports are often associated with non-standard services, peer-to-peer communication, and attack traffic. This group contains 13,265 samples with approximately 29.79% malicious traffic — a 27 percentage point gap representing natural bias.\n\n**Feature Selection — 7 Metadata/Timing Features:** Rather than using all 80 network flow features, the analysis deliberately selects only 7: (1) Protocol — the network protocol type, (2) Flow Duration — total duration of the network flow, (3) Fwd IAT Mean — average inter-arrival time between packets in the forward direction, (4) Bwd IAT Mean — average inter-arrival time in the backward direction, (5) Fwd IAT Std — standard deviation of forward inter-arrival times, (6) Bwd IAT Std — standard deviation of backward inter-arrival times, and (7) High_Port — the binary protected attribute itself. With all 80 features (including packet lengths, byte counts, flag counts, etc.), models achieve 99.9%+ accuracy, making the classification task trivially easy — reweighing cannot shift decision boundaries when models already perfectly separate classes. By limiting to metadata/timing features, Logistic Regression accuracy drops to approximately 90%, creating a challenging enough task where AIF360's Reweighing can meaningfully improve fairness.

In [ ]:
# Clean infinities and NaN in numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('Label')
for col in numeric_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

# Create protected attribute: High_Port (Dst Port > median)
median_port = df['Dst Port'].median()
df['High_Port'] = (df['Dst Port'] > median_port).astype(int)

print('='*70)
print('CREATING PROTECTED ATTRIBUTE: DESTINATION PORT CATEGORY')
print('='*70)
print(f'Dst Port median: {median_port:.0f}')
print(f'\nGroup distribution:')
print(f'  Low port (<=median, privileged):  {(df["High_Port"]==0).sum():,} ({(df["High_Port"]==0).mean()*100:.1f}%)')
print(f'  High port (>median, unprivileged): {(df["High_Port"]==1).sum():,} ({(df["High_Port"]==1).mean()*100:.1f}%)')
print(f'\nMalicious rate by group:')
print(f'  Low port:  {df[df["High_Port"]==0]["Label"].mean()*100:.2f}% malicious')
print(f'  High port: {df[df["High_Port"]==1]["Label"].mean()*100:.2f}% malicious')
print(f'\n  Bias gap: {abs(df[df["High_Port"]==0]["Label"].mean() - df[df["High_Port"]==1]["Label"].mean())*100:.2f} percentage points')

# Select timing/metadata features for classification
feature_cols = ['Protocol', 'Flow Duration', 'Fwd IAT Mean', 'Bwd IAT Mean',
                'Fwd IAT Std', 'Bwd IAT Std', 'High_Port']
df_analysis = df[feature_cols + ['Label']].copy()
print(f'\nSelected features ({len(feature_cols)}): {feature_cols}')
print(f'Analysis dataset shape: {df_analysis.shape}')

This cell creates an AIF360 BinaryLabelDataset from the analysis DataFrame, designating Label as the target with 0 (Benign) as the favorable outcome and 1 (Malicious) as the unfavorable outcome. High_Port is set as the protected attribute, with low-port traffic (value 0) as the privileged group and high-port traffic (value 1) as the unprivileged group. This AIF360 dataset structure enables the computation of group-level fairness metrics throughout the analysis.

In [ ]:
# Create AIF360 BinaryLabelDataset
bld = BinaryLabelDataset(
    favorable_label=0,
    unfavorable_label=1,
    df=df_analysis,
    label_names=['Label'],
    protected_attribute_names=['High_Port']
)

# Define privileged and unprivileged groups
privileged_groups = [{'High_Port': 0}]    # Low port = privileged (lower malicious rate)
unprivileged_groups = [{'High_Port': 1}]  # High port = unprivileged (higher malicious rate)

print(f'Dataset shape: {bld.features.shape}')
print(f'Favorable label: {bld.favorable_label} (Benign)')
print(f'Unfavorable label: {bld.unfavorable_label} (Malicious)')
print(f'Protected attributes: {bld.protected_attribute_names}')

This cell computes dataset-level fairness metrics before any mitigation. The mean difference is -0.2694, meaning high-port traffic receives the favorable outcome (benign) 26.9 percentage points less often than low-port traffic. The disparate impact is 0.7227, well below the 0.8 fairness threshold. Base rates confirm the disparity: low-port traffic has a 97.2% benign rate while high-port traffic has only a 70.2% benign rate. Both metrics trigger bias warnings, confirming significant port-based bias exists in the dataset that needs mitigation.

In [ ]:
# Compute dataset-level fairness metrics BEFORE any mitigation
metric_orig = BinaryLabelDatasetMetric(bld,
                                       unprivileged_groups=unprivileged_groups,
                                       privileged_groups=privileged_groups)

print('=== Dataset-Level Fairness Metrics (Dst Port) ===')
print(f'Mean difference (unprivileged - privileged): {metric_orig.mean_difference():.4f}')
print(f'Disparate impact: {metric_orig.disparate_impact():.4f}')
print(f'\nBase rates:')
print(f'  Overall benign rate: {1 - df_analysis["Label"].mean():.4f}')
print(f'  Low port benign rate: {1 - df_analysis[df_analysis["High_Port"]==0]["Label"].mean():.4f}')
print(f'  High port benign rate: {1 - df_analysis[df_analysis["High_Port"]==1]["Label"].mean():.4f}')

if abs(metric_orig.mean_difference()) > 0.1:
    print(f'\n\u26a0\ufe0f  BIAS DETECTED: Mean difference ({metric_orig.mean_difference():.4f}) exceeds \u00b10.1 threshold')
if metric_orig.disparate_impact() < 0.8 or metric_orig.disparate_impact() > 1.25:
    print(f'\u26a0\ufe0f  BIAS DETECTED: Disparate impact ({metric_orig.disparate_impact():.4f}) outside [0.8, 1.25] range')

## Section 6: Train / Validation / Test Split

This cell splits the AIF360 BinaryLabelDataset into training (70%), validation (15%), and test (15%) sets using AIF360's built-in split method with a fixed random seed for reproducibility. The validation set is used for threshold optimization (finding the best classification threshold that maximizes balanced accuracy), while the test set provides the final unbiased evaluation.

In [ ]:
# Split: 70% train, 15% validation, 15% test
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
dataset_orig_train, dataset_orig_temp = bld.split([0.7], shuffle=True, seed=42)
# Second split: 50/50 of temp -> 15% valid, 15% test
dataset_orig_valid, dataset_orig_test = dataset_orig_temp.split([0.5], shuffle=True, seed=42)

print(f'Training set:   {dataset_orig_train.features.shape[0]:,} samples ({dataset_orig_train.features.shape[0]/bld.features.shape[0]*100:.1f}%)')
print(f'Validation set: {dataset_orig_valid.features.shape[0]:,} samples ({dataset_orig_valid.features.shape[0]/bld.features.shape[0]*100:.1f}%)')
print(f'Test set:       {dataset_orig_test.features.shape[0]:,} samples ({dataset_orig_test.features.shape[0]/bld.features.shape[0]*100:.1f}%)')
print(f'\nTotal: {dataset_orig_train.features.shape[0] + dataset_orig_valid.features.shape[0] + dataset_orig_test.features.shape[0]:,}')

# Check label distribution in each split
for name, ds in [('Train', dataset_orig_train), ('Valid', dataset_orig_valid), ('Test', dataset_orig_test)]:
    mal_rate = (ds.labels.ravel() == 1).mean() * 100
    print(f'{name}: {mal_rate:.1f}% malicious')


## Section 7: Model Training and Evaluation Function

This cell defines the `train_and_evaluate` function, which handles the complete pipeline for training and evaluating each machine learning model. The function scales features using StandardScaler, trains the model with optional sample weights (needed for the reweighing technique), finds the optimal classification threshold by sweeping 100 values on the validation set to maximize balanced accuracy, and evaluates the model on the test set using both AIF360 fairness metrics and standard sklearn metrics (accuracy, precision, recall, F1 score). This reusable function encapsulates the entire model evaluation pipeline and will be called for each of the five classifiers both before and after bias mitigation. No output is produced since this is a function definition.

In [ ]:
def train_and_evaluate(model, model_name, dataset_train, dataset_valid, dataset_test,
                       privileged_groups, unprivileged_groups, sample_weights=None):
    """Train a model, find optimal threshold, evaluate on test set with fairness metrics."""
    # Scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(dataset_train.features)
    y_train = dataset_train.labels.ravel()

    # Train model
    if sample_weights is not None:
        try:
            model.fit(X_train, y_train, sample_weight=sample_weights)
        except TypeError:
            model.fit(X_train, y_train)
    else:
        model.fit(X_train, y_train)

    # Find positive class index
    pos_ind = np.where(model.classes_ == dataset_train.favorable_label)[0][0]

    # Predict probabilities on validation set
    X_valid = scaler.transform(dataset_valid.features)
    valid_pred = dataset_valid.copy(deepcopy=True)

    if hasattr(model, 'predict_proba'):
        valid_pred.scores = model.predict_proba(X_valid)[:, pos_ind].reshape(-1, 1)
    else:
        valid_pred.scores = model.decision_function(X_valid).reshape(-1, 1)
        scores = valid_pred.scores
        valid_pred.scores = (scores - scores.min()) / (scores.max() - scores.min())

    # Find optimal classification threshold on validation set
    num_thresh = 100
    ba_arr = np.zeros(num_thresh)
    class_thresh_arr = np.linspace(0.01, 0.99, num_thresh)

    for idx, class_thresh in enumerate(class_thresh_arr):
        fav_inds = valid_pred.scores > class_thresh
        valid_pred.labels[fav_inds] = dataset_valid.favorable_label
        valid_pred.labels[~fav_inds] = dataset_valid.unfavorable_label
        cm = ClassificationMetric(dataset_valid, valid_pred,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)
        ba_arr[idx] = 0.5 * (cm.true_positive_rate() + cm.true_negative_rate())

    best_idx = np.where(ba_arr == np.max(ba_arr))[0][0]
    best_thresh = class_thresh_arr[best_idx]

    # Evaluate on test set with optimal threshold
    X_test = scaler.transform(dataset_test.features)
    test_pred = dataset_test.copy(deepcopy=True)

    if hasattr(model, 'predict_proba'):
        test_pred.scores = model.predict_proba(X_test)[:, pos_ind].reshape(-1, 1)
    else:
        test_pred.scores = model.decision_function(X_test).reshape(-1, 1)
        scores = test_pred.scores
        test_pred.scores = (scores - scores.min()) / (scores.max() - scores.min())

    fav_inds = test_pred.scores > best_thresh
    test_pred.labels[fav_inds] = dataset_test.favorable_label
    test_pred.labels[~fav_inds] = dataset_test.unfavorable_label

    # Compute metrics
    print(f"\n{'='*60}")
    print(f"  {model_name} (threshold = {best_thresh:.4f})")
    print(f"{'='*60}")
    metrics = compute_metrics(dataset_test, test_pred,
                              unprivileged_groups, privileged_groups, disp=True)

    # Also compute sklearn metrics
    y_true = dataset_test.labels.ravel()
    y_pred = test_pred.labels.ravel()
    print(f"  Accuracy = {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision = {precision_score(y_true, y_pred, pos_label=dataset_test.favorable_label):.4f}")
    print(f"  Recall = {recall_score(y_true, y_pred, pos_label=dataset_test.favorable_label):.4f}")
    print(f"  F1 Score = {f1_score(y_true, y_pred, pos_label=dataset_test.favorable_label):.4f}")

    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['Best threshold'] = best_thresh

    return metrics, scaler, best_thresh


## Section 8: Baseline Modeling - Worked Example

This section establishes the pre-mitigation baseline that students will later use to judge the impact of bias mitigation.


### Guided Example: Train the Baseline Models
Use the completed example below as a reference for the challenge cells later in the notebook.

Workflow to notice:
1. define the model dictionary
2. loop through the models
3. call `train_and_evaluate(...)`
4. store each metric dictionary in `results_before`


In [ ]:
# Guided example: define the five baseline models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVC': SVC(probability=True, random_state=42)
}

# Train and evaluate each model before mitigation
results_before = {}
for name, model in models.items():
    metrics, _, _ = train_and_evaluate(
        model,
        name,
        dataset_orig_train,
        dataset_orig_valid,
        dataset_orig_test,
        privileged_groups,
        unprivileged_groups,
        sample_weights=dataset_orig_train.instance_weights.ravel()
    )
    results_before[name] = metrics

print(f"Stored baseline metrics for {len(results_before)} models.")


### Summary Table — BEFORE Reweighing

### Guided Example: Summarize the Baseline Results
This table is the baseline reference point for every mitigation comparison later in the notebook.


In [ ]:
print('\n' + '=' * 80)
print('SUMMARY: ML Model Fairness Metrics BEFORE Reweighing (Dst Port)')
print('=' * 80)

df_before = pd.DataFrame(results_before).T
df_before.index.name = 'Model'
display(df_before.round(4))


## Section 9: Reweighing - Worked Example

The **Reweighing** algorithm is a preprocessing technique that assigns instance weights to training
samples to equalize favorable outcome rates across privileged and unprivileged groups.

This is the main bias-mitigation tool for the first half of the notebook.


### Guided Example: Apply Reweighing
Study this pattern carefully. Exercise 1 will ask you to quantify how much different models benefited from this mitigation step.


In [ ]:
# Apply Reweighing to the training data
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)
dataset_transf_train = RW.fit_transform(dataset_orig_train)

# Compare dataset-level fairness before and after reweighing
metric_before_rw = BinaryLabelDatasetMetric(
    dataset_orig_train,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)
metric_transf = BinaryLabelDatasetMetric(
    dataset_transf_train,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print('=== Dataset-Level Fairness AFTER Reweighing ===')
print(f'Mean difference BEFORE reweighing: {metric_before_rw.mean_difference():.6f}')
print(f'Mean difference AFTER reweighing:  {metric_transf.mean_difference():.6f}')
print(f'\nDisparate impact BEFORE: {metric_before_rw.disparate_impact():.6f}')
print(f'Disparate impact AFTER:  {metric_transf.disparate_impact():.6f}')

# Show how the instance weights changed
weights = dataset_transf_train.instance_weights
print(f'\nReweighing weights:')
print(f'  Min weight:  {weights.min():.4f}')
print(f'  Max weight:  {weights.max():.4f}')
print(f'  Mean weight: {weights.mean():.4f}')
print(f'  Unique weights: {len(np.unique(weights.round(6)))}')


## Section 10: Models After Reweighing - Worked Example

This section shows what happens when the same five models are retrained on the reweighted dataset.


### Guided Example: Retrain on the Reweighted Data
The code below keeps the model list the same and changes only the training weights. That makes it easier to isolate the effect of reweighing.


In [ ]:
# Guided example: define fresh models for reweighted training
models_rw = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVC': SVC(probability=True, random_state=42)
}

# Train and evaluate each model after mitigation
results_after = {}
for name, model in models_rw.items():
    metrics, _, _ = train_and_evaluate(
        model,
        name,
        dataset_transf_train,
        dataset_orig_valid,
        dataset_orig_test,
        privileged_groups,
        unprivileged_groups,
        sample_weights=dataset_transf_train.instance_weights.ravel()
    )
    results_after[name] = metrics

print(f"Stored post-mitigation metrics for {len(results_after)} models.")


### Summary Table — AFTER Reweighing

### Guided Example: Summarize the Post-Mitigation Results
Compare this table to `df_before` to see which models responded most strongly to reweighing.


In [ ]:
print('\n' + '=' * 80)
print('SUMMARY: ML Model Fairness Metrics AFTER Reweighing (Dst Port)')
print('=' * 80)

df_after = pd.DataFrame(results_after).T
df_after.index.name = 'Model'
display(df_after.round(4))


### Understanding the Fairness Metrics

**Balanced Accuracy** is the average of the true positive rate (correctly catching malicious traffic) and the true negative rate (correctly passing benign traffic). Unlike regular accuracy, it is not skewed by class imbalance and is calculated as (TPR + TNR) / 2. **Statistical Parity Difference (SPD)** measures the difference in the rate of favorable outcomes (predicted benign) between the unprivileged group (high port) and the privileged group (low port), calculated as P(predicted benign | high port) − P(predicted benign | low port). The ideal value is 0 and the fairness threshold is ±0.1; for example, a value of −0.2372 means high-port traffic is predicted benign 23.7 percentage points less often than low-port traffic. **Disparate Impact (DI)** is the ratio version of SPD, calculated as P(predicted benign | high port) / P(predicted benign | low port). The ideal value is 1.0 and the acceptable range is [0.8, 1.25]; a value of 0.748 means high-port traffic receives the favorable outcome at 74.8% the rate of low-port traffic.

**Average Odds Difference** is the average of the false positive rate difference and the true positive rate difference between groups, calculated as [(FPR_unprivileged − FPR_privileged) + (TPR_unprivileged − TPR_privileged)] / 2. The ideal value is 0, and it measures whether the model makes errors equally across both groups. **Equal Opportunity Difference** is the difference in true positive rates between groups, calculated as TPR_unprivileged − TPR_privileged. The ideal value is 0; a value of −0.104 means the model correctly identifies benign high-port traffic 10.4 percentage points less often than benign low-port traffic. **Theil Index** is an individual-level inequality measure borrowed from economics that quantifies how unevenly prediction errors are distributed across all individuals (not just groups). It is calculated as the generalized entropy of the benefit values, with an ideal value of 0 — lower values mean errors are more evenly spread.

**Accuracy** is the standard classification accuracy: the proportion of all predictions that are correct, calculated as (TP + TN) / (TP + TN + FP + FN). **Best Threshold** is the classification probability cutoff that maximizes balanced accuracy on the validation set. Each model outputs a probability for every sample, and if that probability exceeds this threshold the sample is classified as one class versus the other. The threshold is found by sweeping 100 values from 0.01 to 0.99 on the validation set. For example, Logistic Regression's threshold of 0.84 means the model requires high confidence before predicting a sample as malicious.

## Section 11: Before vs After Comparison

Visual comparison of fairness metrics before and after reweighing across all models.

### Guided Example: Visualize Before vs After Reweighing
The chart below helps you see which models improved in fairness and whether that change came with an accuracy cost.


In [ ]:
# Create comparison visualization
model_names = list(results_before.keys())
spd_before = [results_before[m]['Statistical parity difference'] for m in model_names]
spd_after = [results_after[m]['Statistical parity difference'] for m in model_names]
di_before = [results_before[m]['Disparate impact'] for m in model_names]
di_after = [results_after[m]['Disparate impact'] for m in model_names]
acc_before = [results_before[m]['Accuracy'] for m in model_names]
acc_after = [results_after[m]['Accuracy'] for m in model_names]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
x = np.arange(len(model_names))
width = 0.35
ax.bar(x - width / 2, spd_before, width, label='Before', color='#e74c3c', alpha=0.8)
ax.bar(x + width / 2, spd_after, width, label='After', color='#2ecc71', alpha=0.8)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=-0.1, color='orange', linestyle='--', linewidth=1, label='Fairness threshold (+/-0.1)')
ax.axhline(y=0.1, color='orange', linestyle='--', linewidth=1)
ax.set_xlabel('Model')
ax.set_ylabel('Statistical Parity Difference')
ax.set_title('SPD: Before vs After Reweighing', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(x - width / 2, di_before, width, label='Before', color='#e74c3c', alpha=0.8)
ax.bar(x + width / 2, di_after, width, label='After', color='#2ecc71', alpha=0.8)
ax.axhline(y=1.0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=0.8, color='orange', linestyle='--', linewidth=1, label='Fairness threshold (0.8)')
ax.set_xlabel('Model')
ax.set_ylabel('Disparate Impact')
ax.set_title('DI: Before vs After Reweighing', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
ax.legend(fontsize=8)

ax = axes[2]
ax.bar(x - width / 2, acc_before, width, label='Before', color='#3498db', alpha=0.8)
ax.bar(x + width / 2, acc_after, width, label='After', color='#f39c12', alpha=0.8)
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy: Before vs After Reweighing', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
ax.legend(fontsize=8)
ax.set_ylim(0, 1.1)

plt.suptitle('Fairness Metrics Comparison - Before vs After Reweighing (Dst Port)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## Exercise 1: Rank Which Models Benefited Most from Reweighing

Use the baseline and post-mitigation results to build a model scorecard.

Learning goal:
- measure how strongly different algorithms respond to the same preprocessing mitigation step

You are not writing this from scratch. Use the Logistic Regression example row below, then uncomment the full reference solution for the remaining models.


In [ ]:
# Exercise 1: build a reweighing scorecard for all five models.

reweighing_rows = [
    {
        'Model': 'Logistic Regression',
        'SPD Before': results_before['Logistic Regression']['Statistical parity difference'],
        'SPD After': results_after['Logistic Regression']['Statistical parity difference'],
        'DI Before': results_before['Logistic Regression']['Disparate impact'],
        'DI After': results_after['Logistic Regression']['Disparate impact'],
        'Accuracy Before': results_before['Logistic Regression']['Accuracy'],
        'Accuracy After': results_after['Logistic Regression']['Accuracy']
    },
    # Uncomment this completed Random Forest row.
    # {
    #     'Model': 'Random Forest',
    #     'SPD Before': results_before['Random Forest']['Statistical parity difference'],
    #     'SPD After': results_after['Random Forest']['Statistical parity difference'],
    #     'DI Before': results_before['Random Forest']['Disparate impact'],
    #     'DI After': results_after['Random Forest']['Disparate impact'],
    #     'Accuracy Before': results_before['Random Forest']['Accuracy'],
    #     'Accuracy After': results_after['Random Forest']['Accuracy']
    # },
    # Uncomment this completed Decision Tree row.
    # {
    #     'Model': 'Decision Tree',
    #     'SPD Before': results_before['Decision Tree']['Statistical parity difference'],
    #     'SPD After': results_after['Decision Tree']['Statistical parity difference'],
    #     'DI Before': results_before['Decision Tree']['Disparate impact'],
    #     'DI After': results_after['Decision Tree']['Disparate impact'],
    #     'Accuracy Before': results_before['Decision Tree']['Accuracy'],
    #     'Accuracy After': results_after['Decision Tree']['Accuracy']
    # },
    # Uncomment this completed KNN row.
    # {
    #     'Model': 'KNN',
    #     'SPD Before': results_before['KNN']['Statistical parity difference'],
    #     'SPD After': results_after['KNN']['Statistical parity difference'],
    #     'DI Before': results_before['KNN']['Disparate impact'],
    #     'DI After': results_after['KNN']['Disparate impact'],
    #     'Accuracy Before': results_before['KNN']['Accuracy'],
    #     'Accuracy After': results_after['KNN']['Accuracy']
    # },
    # Uncomment this completed SVC row.
    # {
    #     'Model': 'SVC',
    #     'SPD Before': results_before['SVC']['Statistical parity difference'],
    #     'SPD After': results_after['SVC']['Statistical parity difference'],
    #     'DI Before': results_before['SVC']['Disparate impact'],
    #     'DI After': results_after['SVC']['Disparate impact'],
    #     'Accuracy Before': results_before['SVC']['Accuracy'],
    #     'Accuracy After': results_after['SVC']['Accuracy']
    # },
]

reweighing_scorecard = None

# TODO 1: create the DataFrame
# reweighing_scorecard = pd.DataFrame(reweighing_rows)

# TODO 2: add the derived columns below
# reweighing_scorecard['Abs SPD Improvement'] = (
#     reweighing_scorecard['SPD Before'].abs() - reweighing_scorecard['SPD After'].abs()
# )
# reweighing_scorecard['DI Change'] = (
#     reweighing_scorecard['DI After'] - reweighing_scorecard['DI Before']
# )
# reweighing_scorecard['Accuracy Change'] = (
#     reweighing_scorecard['Accuracy After'] - reweighing_scorecard['Accuracy Before']
# )

# TODO 3: sort by largest absolute SPD improvement first
# reweighing_scorecard = reweighing_scorecard.sort_values('Abs SPD Improvement', ascending=False)

if reweighing_scorecard is not None and len(reweighing_rows) == 5:
    display(reweighing_scorecard.round(4))
    best_model = reweighing_scorecard.iloc[0]['Model']
    print(f"Model with the strongest fairness improvement after reweighing: {best_model}")
else:
    print(
        "Uncomment the remaining four model rows and the reference solution lines for `reweighing_scorecard`, "
        "and rerun this cell."
    )


## Section 12: Postprocessing - Reject Option Classification (Worked Example)

This section introduces a second mitigation strategy that acts **after** the model has already made predictions.


### Guided Example: Reject Option Classification
Run this provided example to compare baseline Logistic Regression with a postprocessing fairness intervention.

Pay attention to:
1. the baseline Logistic Regression metrics
2. the postprocessed metrics after Reject Option Classification
3. the fairness-versus-accuracy tradeoff

You will use the resulting variables in Exercise 2.


In [ ]:
from aif360.algorithms.postprocessing import RejectOptionClassification

# ---- Step 1: Train baseline Logistic Regression on ORIGINAL data ----
print('='*60)
print('  Baseline Logistic Regression (before any mitigation)')
print('='*60)

scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(dataset_orig_train.features)
y_train_lr = dataset_orig_train.labels.ravel()

lr_baseline = LogisticRegression(max_iter=1000, random_state=42)
lr_baseline.fit(X_train_lr, y_train_lr)

pos_ind_lr = np.where(lr_baseline.classes_ == dataset_orig_train.favorable_label)[0][0]

# Threshold sweep on validation set
X_valid_lr = scaler_lr.transform(dataset_orig_valid.features)
valid_pred_lr = dataset_orig_valid.copy(deepcopy=True)
valid_pred_lr.scores = lr_baseline.predict_proba(X_valid_lr)[:, pos_ind_lr].reshape(-1, 1)

num_thresh = 100
ba_arr = np.zeros(num_thresh)
class_thresh_arr = np.linspace(0.01, 0.99, num_thresh)
for idx, ct in enumerate(class_thresh_arr):
    valid_pred_lr.labels[valid_pred_lr.scores > ct] = dataset_orig_valid.favorable_label
    valid_pred_lr.labels[valid_pred_lr.scores <= ct] = dataset_orig_valid.unfavorable_label
    cm_v = ClassificationMetric(dataset_orig_valid, valid_pred_lr,
                                unprivileged_groups=unprivileged_groups,
                                privileged_groups=privileged_groups)
    ba_arr[idx] = 0.5 * (cm_v.true_positive_rate() + cm_v.true_negative_rate())

best_thresh_lr = class_thresh_arr[np.argmax(ba_arr)]
print(f'Best threshold: {best_thresh_lr:.4f}')

# Final validation predictions at best threshold (needed for ROC fitting)
valid_pred_lr_final = dataset_orig_valid.copy(deepcopy=True)
valid_pred_lr_final.scores = lr_baseline.predict_proba(X_valid_lr)[:, pos_ind_lr].reshape(-1, 1)
valid_pred_lr_final.labels[valid_pred_lr_final.scores > best_thresh_lr] = dataset_orig_valid.favorable_label
valid_pred_lr_final.labels[valid_pred_lr_final.scores <= best_thresh_lr] = dataset_orig_valid.unfavorable_label

# Test predictions
X_test_lr = scaler_lr.transform(dataset_orig_test.features)
test_pred_lr_baseline = dataset_orig_test.copy(deepcopy=True)
test_pred_lr_baseline.scores = lr_baseline.predict_proba(X_test_lr)[:, pos_ind_lr].reshape(-1, 1)
test_pred_lr_baseline.labels[test_pred_lr_baseline.scores > best_thresh_lr] = dataset_orig_test.favorable_label
test_pred_lr_baseline.labels[test_pred_lr_baseline.scores <= best_thresh_lr] = dataset_orig_test.unfavorable_label

print('\nBaseline LR metrics on test set:')
metrics_lr_baseline = compute_metrics(dataset_orig_test, test_pred_lr_baseline,
                                       unprivileged_groups, privileged_groups, disp=True)
y_true_test = dataset_orig_test.labels.ravel()
acc_baseline = accuracy_score(y_true_test, test_pred_lr_baseline.labels.ravel())
metrics_lr_baseline['Accuracy'] = acc_baseline
print(f'  Accuracy = {acc_baseline:.4f}')

# ---- Step 2: Apply Reject Option Classification ----
print(f'\n{"="*60}')
print('  Reject Option Classification (Postprocessing)')
print(f'{"="*60}')

roc = RejectOptionClassification(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
    low_class_thresh=0.01, high_class_thresh=0.99,
    num_class_thresh=100, num_ROC_margin=50,
    metric_name='Statistical parity difference',
    metric_ub=0.05, metric_lb=-0.05
)

# Fit on validation set, predict on test set
roc = roc.fit(dataset_orig_valid, valid_pred_lr_final)
test_pred_roc = roc.predict(test_pred_lr_baseline)

print('\nPostprocessed LR metrics on test set:')
metrics_lr_roc = compute_metrics(dataset_orig_test, test_pred_roc,
                                  unprivileged_groups, privileged_groups, disp=True)
acc_roc = accuracy_score(y_true_test, test_pred_roc.labels.ravel())
metrics_lr_roc['Accuracy'] = acc_roc
print(f'  Accuracy = {acc_roc:.4f}')

# ---- Before vs After summary ----
print(f'\n{"="*60}')
print('  Before vs After Postprocessing')
print(f'{"="*60}')
print(f'{"Metric":<35} {"Before":>10} {"After":>10} {"Change":>10}')
print('-'*65)
for k in ['Statistical parity difference', 'Disparate impact', 'Average odds difference',
          'Equal opportunity difference', 'Balanced accuracy', 'Accuracy']:
    b = metrics_lr_baseline[k]
    a = metrics_lr_roc[k]
    print(f'  {k:<33} {b:>10.4f} {a:>10.4f} {a-b:>+10.4f}')


## Exercise 2: Compare Preprocessing vs Postprocessing for Logistic Regression

This challenge compares two different mitigation strategies for the same model:
- **Preprocessing**: Reweighing
- **Postprocessing**: Reject Option Classification

Learning goal:
- compare where each mitigation acts and how that changes fairness and accuracy


### Exercise 2 Instructions
Use the baseline row below as your template.

Complete the TODOs to:
1. add the preprocessing and postprocessing rows
2. build a comparison DataFrame
3. calculate fairness and accuracy change columns
4. create a compact three-bar comparison figure

A full reference solution is included as commented code in the next cell.


In [ ]:
# Exercise 2: compare Logistic Regression across three mitigation settings.

strategy_rows = [
    {
        'Approach': 'Baseline (No Mitigation)',
        'SPD': metrics_lr_baseline['Statistical parity difference'],
        'DI': metrics_lr_baseline['Disparate impact'],
        'Accuracy': metrics_lr_baseline['Accuracy']
    },
    # Uncomment this completed preprocessing row.
    # {
    #     'Approach': 'Preprocessing (Reweighing)',
    #     'SPD': results_after['Logistic Regression']['Statistical parity difference'],
    #     'DI': results_after['Logistic Regression']['Disparate impact'],
    #     'Accuracy': results_after['Logistic Regression']['Accuracy']
    # },
    # Uncomment this completed postprocessing row.
    # {
    #     'Approach': 'Postprocessing (ROC)',
    #     'SPD': metrics_lr_roc['Statistical parity difference'],
    #     'DI': metrics_lr_roc['Disparate impact'],
    #     'Accuracy': metrics_lr_roc['Accuracy']
    # }
]

strategy_scorecard = None

# Uncomment the full reference solution below.
# TODO 1: create the DataFrame
# strategy_scorecard = pd.DataFrame(strategy_rows)

# TODO 2: add these derived columns
# baseline_spd = abs(metrics_lr_baseline['Statistical parity difference'])
# baseline_acc = metrics_lr_baseline['Accuracy']
# strategy_scorecard['Abs SPD Improvement vs Baseline'] = baseline_spd - strategy_scorecard['SPD'].abs()
# strategy_scorecard['Accuracy Change vs Baseline'] = strategy_scorecard['Accuracy'] - baseline_acc
# strategy_scorecard['DI Distance From Ideal'] = (strategy_scorecard['DI'] - 1.0).abs()

if strategy_scorecard is not None and len(strategy_rows) == 3:
    display(strategy_scorecard.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ['#e74c3c', '#3498db', '#2ecc71']

    axes[0].bar(strategy_scorecard['Approach'], strategy_scorecard['SPD'], color=colors, alpha=0.85)
    axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.6)
    axes[0].set_title('Statistical Parity Difference')
    axes[0].set_ylabel('SPD')

    axes[1].bar(strategy_scorecard['Approach'], strategy_scorecard['DI'], color=colors, alpha=0.85)
    axes[1].axhline(y=1.0, color='black', linestyle='--', alpha=0.6)
    axes[1].set_title('Disparate Impact')
    axes[1].set_ylabel('DI')

    axes[2].bar(strategy_scorecard['Approach'], strategy_scorecard['Accuracy'], color=colors, alpha=0.85)
    axes[2].set_title('Accuracy')
    axes[2].set_ylabel('Accuracy')
    axes[2].set_ylim(0.7, 1.0)

    for ax in axes:
        ax.tick_params(axis='x', rotation=15)

    plt.suptitle('Logistic Regression: Mitigation Strategy Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    fairest_row = strategy_scorecard.loc[strategy_scorecard['SPD'].abs().idxmin()]
    print(f"Fairest approach by |SPD|: {fairest_row['Approach']}")
else:
    print(
        "Add the preprocessing and postprocessing rows, create `strategy_scorecard`, "
        "and rerun this cell."
    )


## Section 14: Final Written Response


Answer the question below in your own words. Write your response in the markdown cell that follows this prompt.

**Comprehensive question:** In this network attack-detection module, which mitigation strategy would you recommend for deployment: no mitigation, preprocessing with reweighing, or postprocessing with Reject Option Classification? Support your answer with at least two fairness metrics and one accuracy value. Also explain how your choice affects the tradeoff between catching attacks and reducing bias against high-port traffic.


**Write your response here.**

Replace this text with a 6-8 sentence answer.

Your response should include:
- the mitigation strategy you recommend
- at least two fairness metrics
- one accuracy value
- a tradeoff discussion about fairness versus attack detection performance
- a short justification for why your recommendation fits this security setting
